In [ ]:
sys.path.append("/home/ubuntu/iti_env/benchmarking/ITI_project")
import time
import numpy as np
import torch
import igraph as ig
from pathlib import Path
from torch_geometric.datasets import Planetoid, CitationFull
from torch_geometric.nn import GCN, GAT, GIN, GraphSAGE
import sys, time, contextlib
from torch_geometric.datasets import Planetoid, CitationFull
import inspect

import src.optimize as opt
import src.map_equation as meq
from src.utils import compare_partitions

import src.neuromap as nm
from src.utils import atomic_write_json, append_csv_row, load_json, trial_already_done

# reuse the WikiCS script's generic helpers instead of redefining them
from run_neuromap import sparse_from_igraph, to_dataset, Neuromap_with_loss_tracking

In [ ]:
HYPERPARAMS_CITATION = {
    "hidden_channels": 100,
    "num_layers": 2,
    "act": "selu",
    "norm": "batch",
    "dropout": 0.1,
    "lr": 1e-3,
    "epochs": 1000,
    "patience": 100,
}
OUT_CHANNELS_CITATION = 400
NUM_RERUNS_CITATION = 10
RESULTS_ROOT_CITATION = "/content/drive/My Drive/results_citation"

In [ ]:
def load_cora_graph(directed: bool) -> ig.Graph:
    dataset = Planetoid(root="/content/drive/My Drive/data/Cora", name="Cora")
    data = dataset[0]
    edges = list(zip(data.edge_index[0].tolist(), data.edge_index[1].tolist()))
    g_full = ig.Graph(n=data.num_nodes, edges=edges, directed=directed)
    if not directed:
        g_full = g_full.simplify()
    components = g_full.connected_components(mode="weak") if directed else g_full.connected_components()
    giant_idx = max(components, key=len)
    return g_full.induced_subgraph(giant_idx)


def load_coraml_graph(directed: bool) -> ig.Graph:
    dataset = CitationFull(root="/content/drive/My Drive/data/CoraML", name="Cora_ML", to_undirected=False)
    data = dataset[0]
    edges = list(zip(data.edge_index[0].tolist(), data.edge_index[1].tolist()))
    g_full = ig.Graph(n=data.num_nodes, edges=edges, directed=True)
    if not directed:
        g_full = g_full.as_undirected(combine_edges="first")
    components = g_full.connected_components(mode="weak") if g_full.is_directed() else g_full.connected_components()
    giant_idx = max(components, key=len)
    return g_full.induced_subgraph(giant_idx)


DATASETS_CITATION = {"Cora": load_cora_graph, "Cora_ML": load_coraml_graph}

In [ ]:
def build_model_citation(architecture: str, n: int) -> torch.nn.Module:
    model_cls = ARCHITECTURES[architecture]
    return model_cls(
        in_channels=n,
        hidden_channels=HYPERPARAMS_CITATION["hidden_channels"],
        num_layers=HYPERPARAMS_CITATION["num_layers"],
        out_channels=OUT_CHANNELS_CITATION,
        act=HYPERPARAMS_CITATION["act"],
        norm=HYPERPARAMS_CITATION["norm"],
        dropout=HYPERPARAMS_CITATION["dropout"],
    )

In [ ]:
import sys
sys.path.append("/content/drive/My Drive/")

import map_equation as meq

def run_single_citation(g: ig.Graph, arch_name: str, rerun_idx: int, out_root: str) -> dict:
    run_dir = f"{out_root}/{arch_name}/run_{rerun_idx:02d}"
    Path(run_dir).mkdir(parents=True, exist_ok=True)
    assignment_path = f"{run_dir}/cluster_assignment.json"
    loss_curve_path = f"{run_dir}/loss_curve.json"

    if trial_already_done(assignment_path):
        cached = load_json(assignment_path)
        return {"final_loss": cached["final_loss"], "recomputed_L": cached.get("recomputed_L"),
                "runtime_seconds": cached["runtime_seconds"], "assignment_path": assignment_path,
                "loss_curve_path": loss_curve_path, "cached": True}

    n = g.vcount()
    data = to_dataset(G=g, y_true=[])
    model = build_model_citation(arch_name, n)
    neuromap = Neuromap_with_loss_tracking(model=model, device=DEVICE)

    t0 = time.perf_counter()
    final_loss, S, loss_curve = neuromap.fit(data, epochs=HYPERPARAMS_CITATION["epochs"],
                                              patience=HYPERPARAMS_CITATION["patience"],
                                              lr=HYPERPARAMS_CITATION["lr"])
    t_elapsed = time.perf_counter() - t0

    hard_clusters = nm.get_hard_clusters(S) if S is not None else []
    recomputed_L = meq.compute_description_length(g, hard_clusters) if hard_clusters else None

    atomic_write_json(assignment_path, {
        "status": "completed", "architecture": arch_name, "rerun": rerun_idx,
        "final_loss": final_loss, "recomputed_L": recomputed_L,
        "num_communities": int(len(set(hard_clusters))) if hard_clusters else 0,
        "runtime_seconds": t_elapsed, "communities": hard_clusters,
    })
    atomic_write_json(loss_curve_path, {"status": "completed", "architecture": arch_name,
                                         "rerun": rerun_idx, "loss_curve": loss_curve})

    return {"final_loss": final_loss, "recomputed_L": recomputed_L, "runtime_seconds": t_elapsed,
            "assignment_path": assignment_path, "loss_curve_path": loss_curve_path, "cached": False}

In [ ]:
def run_variant_citation(dataset_name: str, directed: bool):
    variant = "directed" if directed else "undirected"
    print(f"\n=== {dataset_name} ({variant}) ===")
    g = DATASETS_CITATION[dataset_name](directed=directed)
    print(f"Loaded GCC: {g.vcount()} nodes, {g.ecount()} edges, directed={g.is_directed()}")

    variant_dir = f"{RESULTS_ROOT_CITATION}/{dataset_name}/{variant}"
    Path(variant_dir).mkdir(parents=True, exist_ok=True)
    overview_path = f"{variant_dir}/overview.csv"
    overview_fields = ["architecture", "rerun", "hidden_channels", "num_layers", "out_channels",
                        "dropout", "lr", "epochs", "patience", "final_loss", "recomputed_L",
                        "runtime_seconds", "cluster_assignment_file", "loss_curve_file", "cached"]

    for arch_name in ARCHITECTURES:
        print(f"\n-- {arch_name} --")
        for rerun in range(NUM_RERUNS_CITATION):
            result = run_single_citation(g, arch_name, rerun, variant_dir)
            append_csv_row(overview_path, overview_fields, {
                "architecture": arch_name, "rerun": rerun,
                "hidden_channels": HYPERPARAMS_CITATION["hidden_channels"],
                "num_layers": HYPERPARAMS_CITATION["num_layers"],
                "out_channels": OUT_CHANNELS_CITATION,
                "dropout": HYPERPARAMS_CITATION["dropout"], "lr": HYPERPARAMS_CITATION["lr"],
                "epochs": HYPERPARAMS_CITATION["epochs"], "patience": HYPERPARAMS_CITATION["patience"],
                "final_loss": result["final_loss"], "recomputed_L": result["recomputed_L"],
                "runtime_seconds": result["runtime_seconds"], "cluster_assignment_file": result["assignment_path"],
                "loss_curve_file": result["loss_curve_path"], "cached": result["cached"],
            })
            tag = " (cached)" if result["cached"] else ""
            print(f"  rerun {rerun:02d}: L = {result['final_loss']:.6f} bits "
                  f"(recomputed: {result['recomputed_L']}), runtime = {result['runtime_seconds']:.1f}s{tag}")

In [ ]:
Path(RESULTS_ROOT_CITATION).mkdir(parents=True, exist_ok=True)

run_variant_citation("Cora", directed=False)
run_variant_citation("Cora_ML", directed=True)


=== Cora (undirected) ===


Processing...
Done!


Loaded GCC: 2485 nodes, 5069 edges, directed=False

-- GCN --


/usr/local/lib/python3.12/dist-packages/torch_geometric/nn/conv/gcn_conv.py:274: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)


[Epoch    0] L = 12.71079636 bits
[Epoch    1] L = 12.57879925 bits
[Epoch    2] L = 12.28793240 bits
[Epoch    3] L = 11.83828735 bits
[Epoch    4] L = 11.33487225 bits
[Epoch    5] L = 10.87833691 bits
[Epoch    6] L = 10.43992615 bits
[Epoch    7] L = 10.07568741 bits
[Epoch    8] L = 9.75696659 bits
[Epoch    9] L = 9.48915291 bits
[Epoch   10] L = 9.25499058 bits
[Epoch   11] L = 9.03889084 bits
[Epoch   12] L = 8.84529305 bits
[Epoch   13] L = 8.67248058 bits
[Epoch   14] L = 8.51510429 bits
[Epoch   15] L = 8.35708332 bits
[Epoch   16] L = 8.23181438 bits
[Epoch   17] L = 8.12177277 bits
[Epoch   18] L = 8.02177048 bits
[Epoch   19] L = 7.93250179 bits
[Epoch   20] L = 7.85339499 bits
[Epoch   21] L = 7.77741385 bits
[Epoch   22] L = 7.71297550 bits
[Epoch   23] L = 7.64875650 bits
[Epoch   24] L = 7.59442902 bits
[Epoch   25] L = 7.54586124 bits
[Epoch   26] L = 7.50463343 bits
[Epoch   27] L = 7.46592426 bits
[Epoch   28] L = 7.43140888 bits
[Epoch   29] L = 7.40171909 bits
[E

...

Streaming output truncated to the last 5000 lines.
[Epoch  469] L = 7.58076668 bits
[Epoch  470] L = 7.58072519 bits
[Epoch  471] L = 7.58073616 bits
[Epoch  472] L = 7.58074379 bits
[Epoch  473] L = 7.58075762 bits
[Epoch  474] L = 7.58071661 bits
[Epoch  475] L = 7.58073902 bits
[Epoch  476] L = 7.58070993 bits
[Epoch  477] L = 7.58071756 bits
[Epoch  478] L = 7.58081007 bits
[Epoch  479] L = 7.58071804 bits
[Epoch  480] L = 7.58071804 bits
[Epoch  481] L = 7.58078098 bits
[Epoch  482] L = 7.58072281 bits
[Epoch  483] L = 7.58075380 bits
[Epoch  484] L = 7.58069706 bits
[Epoch  485] L = 7.58070326 bits
[Epoch  486] L = 7.58073473 bits
[Epoch  487] L = 7.58073425 bits
[Epoch  488] L = 7.58074474 bits
[Epoch  489] L = 7.58065510 bits
[Epoch  490] L = 7.58070421 bits
[Epoch  491] L = 7.58068085 bits
[Epoch  492] L = 7.58069801 bits
[Epoch  493] L = 7.58068991 bits
[Epoch  494] L = 7.58068848 bits
[Epoch  495] L = 7.58067226 bits
[Epoch  496] L = 7.58071899 bits
[Epoch  497] L = 7.580692

/usr/local/lib/python3.12/dist-packages/torch_geometric/nn/conv/sage_conv.py:152: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x[0], reduce=self.aggr)


Streaming output truncated to the last 5000 lines.
[Epoch    7] L = 8.33761597 bits
[Epoch    8] L = 8.12033844 bits
[Epoch    9] L = 7.95880556 bits
[Epoch   10] L = 7.84883213 bits
[Epoch   11] L = 7.76378059 bits
[Epoch   12] L = 7.68940544 bits
[Epoch   13] L = 7.62809753 bits
[Epoch   14] L = 7.57445192 bits
[Epoch   15] L = 7.53057718 bits
[Epoch   16] L = 7.48604774 bits
[Epoch   17] L = 7.45631981 bits
[Epoch   18] L = 7.42571402 bits
[Epoch   19] L = 7.40389729 bits
[Epoch   20] L = 7.38210106 bits
[Epoch   21] L = 7.36390829 bits
[Epoch   22] L = 7.34546471 bits
[Epoch   23] L = 7.32657146 bits
[Epoch   24] L = 7.31404209 bits
[Epoch   25] L = 7.29717255 bits
[Epoch   26] L = 7.28809929 bits
[Epoch   27] L = 7.28159189 bits
[Epoch   28] L = 7.27556038 bits
[Epoch   29] L = 7.27048588 bits
[Epoch   30] L = 7.26301193 bits
[Epoch   31] L = 7.25922441 bits
[Epoch   32] L = 7.25465584 bits
[Epoch   33] L = 7.25047255 bits
[Epoch   34] L = 7.24757671 bits
[Epoch   35] L = 7.246203

Processing...
Done!


Streaming output truncated to the last 5000 lines.
[Epoch    5] L = 4.49396324 bits
[Epoch    6] L = 4.31895208 bits
[Epoch    7] L = 4.12590313 bits
[Epoch    8] L = 3.99051905 bits
[Epoch    9] L = 3.87855959 bits
[Epoch   10] L = 3.78209400 bits
[Epoch   11] L = 3.68604088 bits
[Epoch   12] L = 3.62560940 bits
[Epoch   13] L = 3.56789017 bits
[Epoch   14] L = 3.53555346 bits
[Epoch   15] L = 3.50673389 bits
[Epoch   16] L = 3.48086739 bits
[Epoch   17] L = 3.45775080 bits
[Epoch   18] L = 3.43952370 bits
[Epoch   19] L = 3.41930342 bits
[Epoch   20] L = 3.40124035 bits
[Epoch   21] L = 3.38521004 bits
[Epoch   22] L = 3.37278128 bits
[Epoch   23] L = 3.35896778 bits
[Epoch   24] L = 3.34984159 bits
[Epoch   25] L = 3.34090519 bits
[Epoch   26] L = 3.32839727 bits
[Epoch   27] L = 3.31748390 bits
[Epoch   28] L = 3.30568171 bits
[Epoch   29] L = 3.29170132 bits
[Epoch   30] L = 3.28176022 bits
[Epoch   31] L = 3.27226830 bits
[Epoch   32] L = 3.26282692 bits
[Epoch   33] L = 3.258430